In [2]:
import numpy as np
import os
import warnings
import pywt
from scipy import signal
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

warnings.filterwarnings("ignore")

focal_folder_path = '/Users/gaurav/Downloads/Complete set/Data_F_Ind_1_750'
non_focal_folder_path = '/Users/gaurav/Downloads/Complete set/Data_N_Ind_1_750'

def sevcik_fractal_dimension(data):
    L = np.sum(np.sqrt(1 + np.diff(data)**2))
    return L / len(data)

def wavelet_features(data, wavelet='db4', level=4):
    coeffs = pywt.wavedec(data, wavelet, level=level)
    features = [np.mean(c) for c in coeffs] + [np.std(c) for c in coeffs]
    return features

def hurst_exponent(data):
    N = len(data)
    T = np.arange(1, N + 1)
    Y = np.cumsum(data - np.mean(data))
    R = np.max(Y) - np.min(Y)
    S = np.std(data)
    return R / S if S != 0 else 0

def higuchi_fractal_dimension(data, kmax=10):
    Lk = []
    N = len(data)
    for k in range(1, kmax + 1):
        Lmk = []
        for m in range(k):
            Lmki = 0
            for i in range(1, int((N - m) / k)):
                Lmki += abs(data[m + i * k] - data[m + (i - 1) * k])
            norm_factor = (N - 1) / (int((N - m) / k) * k)
            Lmk.append(Lmki * norm_factor)
        Lk.append(np.mean(Lmk))
    hfd = np.polyfit(np.log(range(1, kmax + 1)), np.log(Lk), 1)[0]
    return hfd

def katz_fractal_dimension(data):
    d = np.max(np.sqrt((np.arange(len(data)) - 0)**2 + (data - data[0])**2))
    L = np.sum(np.sqrt(1 + np.diff(data)**2))
    return np.log10(L) / (np.log10(d) + np.log10(len(data)))

def power_spectral_density(data):
    freqs, psd = signal.welch(data, fs=512)
    return np.mean(psd)

def spectral_entropy(data):
    power_spectrum = np.abs(np.fft.fft(data))**2
    power_spectrum /= np.sum(power_spectrum)
    return -np.sum(power_spectrum * np.log2(power_spectrum + 1e-10))

def bandpower(data, sf, band, window_sec=None):
    band = np.asarray(band)
    low, high = band
    if window_sec:
        nperseg = int(window_sec * sf)
    else:
        nperseg = None
    freqs, psd = signal.welch(data, sf, nperseg=nperseg)
    freq_res = freqs[1] - freqs[0]
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    bp = np.trapz(psd[idx_band], dx=freq_res)
    return bp

def svd_features(data):
    u, s, vh = np.linalg.svd(data.reshape(-1, 1), full_matrices=False)
    return np.mean(s), np.std(s)

def dmd_features(data, rank=5):
    x = data[:-1].reshape(-1, 1)
    y = data[1:].reshape(-1, 1)
    u, s, vh = np.linalg.svd(x, full_matrices=False)
    s = np.diag(s)
    u_r = u[:, :rank]
    s_r = s[:rank, :rank]
    vh_r = vh[:rank, :]
    A_tilde = u_r.T @ y @ vh_r.T @ np.linalg.inv(s_r)
    eigenvalues = np.linalg.eigvals(A_tilde)
    return np.real(eigenvalues).mean(), np.imag(eigenvalues).mean()

def load_data_with_features(folder_path, limit=3750):
    file_list = [f for f in os.listdir(folder_path) if f.endswith('.txt')][:limit]
    data_array = []
    for file_name in file_list:
        file_path = os.path.join(folder_path, file_name)
        data = np.loadtxt(file_path, delimiter=',').flatten()
        
        hfd = higuchi_fractal_dimension(data)
        kfd = katz_fractal_dimension(data)
        psd = power_spectral_density(data)
        se = spectral_entropy(data)
        sevcik = sevcik_fractal_dimension(data)
        hurst = hurst_exponent(data)
        wavelet_feat = wavelet_features(data)
        mean_val, std_val = np.mean(data), np.std(data)
        var_val, skew_val, kurt_val = np.var(data), skew(data), kurtosis(data)
        rms_val = np.sqrt(np.mean(data**2))
        zcr_val = ((data[:-1] * data[1:]) < 0).sum() / len(data)
        mean_svd, std_svd = svd_features(data)
        real_dmd, imag_dmd = dmd_features(data)
        sf = 250
        delta_bp = bandpower(data, sf, [0.5, 4])
        theta_bp = bandpower(data, sf, [4, 8])
        alpha_bp = bandpower(data, sf, [8, 12])
        beta_bp = bandpower(data, sf, [12, 30])
        gamma_bp = bandpower(data, sf, [30, 100])
        
        features = [
            hfd, kfd, psd, se, sevcik, hurst,
            delta_bp, theta_bp, alpha_bp, beta_bp, gamma_bp,
            mean_val, std_val, var_val, skew_val, kurt_val,
            rms_val, zcr_val, mean_svd, std_svd, real_dmd, imag_dmd
        ] + wavelet_feat
        data_array.append(features)
    return np.array(data_array)

focal_data = load_data_with_features(focal_folder_path)
non_focal_data = load_data_with_features(non_focal_folder_path)

focal_labels = np.ones(len(focal_data))
non_focal_labels = np.zeros(len(non_focal_data))
X = np.vstack((focal_data, non_focal_data))
y = np.concatenate((focal_labels, non_focal_labels))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.06, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

def evaluate_model_random_search(model, params, model_name, n_iter=4):
    random_search = RandomizedSearchCV(
        model, param_distributions=params, n_iter=n_iter, cv=5, scoring='accuracy', n_jobs=-1, random_state=42
    )
    random_search.fit(X_train_balanced, y_train_balanced)
    y_pred = random_search.predict(X_test_scaled)
    print(f"{model_name} Best Params:", random_search.best_params_)
    print(f"{model_name} Report:")
    print(classification_report(y_test, y_pred))
    print(f"{model_name} Accuracy:", accuracy_score(y_test, y_pred))


In [16]:
gb_params = {
    'n_estimators': [600], 'learning_rate': [0.2, 0.25], 'max_depth': [13],
    'min_samples_split': [5], 'min_samples_leaf': [10], 'subsample': [1.0],
    'max_features': ['log2']
}
evaluate_model_random_search(GradientBoostingClassifier(), gb_params, 'Gradient Boosting')


Gradient Boosting Best Params: {'subsample': 1.0, 'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_features': 'log2', 'max_depth': 13, 'learning_rate': 0.25}
Gradient Boosting Report:
              precision    recall  f1-score   support

         0.0       0.89      0.89      0.89       209
         1.0       0.90      0.90      0.90       241

    accuracy                           0.89       450
   macro avg       0.89      0.89      0.89       450
weighted avg       0.89      0.89      0.89       450

Gradient Boosting Accuracy: 0.8933333333333333


In [18]:
xgboost_params = {
    'n_estimators': [800], 'learning_rate': [0.15], 'max_depth': [13],
    'min_child_weight': [4], 'subsample': [1], 'colsample_bytree': [1.0]
}
evaluate_model_random_search(XGBClassifier(eval_metric='logloss'), xgboost_params, 'XGBoost')


XGBoost Best Params: {'subsample': 1, 'n_estimators': 800, 'min_child_weight': 4, 'max_depth': 13, 'learning_rate': 0.15, 'colsample_bytree': 1.0}
XGBoost Report:
              precision    recall  f1-score   support

         0.0       0.88      0.89      0.88       209
         1.0       0.90      0.90      0.90       241

    accuracy                           0.89       450
   macro avg       0.89      0.89      0.89       450
weighted avg       0.89      0.89      0.89       450

XGBoost Accuracy: 0.8911111111111111


In [7]:
rf_params = {
    'n_estimators': [300], 'max_depth': [20], 'min_samples_split': [2],
    'min_samples_leaf': [2], 'max_features': ['log2'], 'bootstrap': [False]
}
evaluate_model_random_search(RandomForestClassifier(), rf_params, 'Random Forest')


Random Forest Best Params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 20, 'bootstrap': False}
Random Forest Report:
              precision    recall  f1-score   support

         0.0       0.87      0.85      0.86       209
         1.0       0.87      0.89      0.88       241

    accuracy                           0.87       450
   macro avg       0.87      0.87      0.87       450
weighted avg       0.87      0.87      0.87       450

Random Forest Accuracy: 0.8711111111111111


In [8]:
svm_params = [{'kernel': ['rbf'], 'C': [10, 50], 'gamma': ['scale', 0.1, 1]}]
evaluate_model_random_search(SVC(), svm_params, 'SVM')


SVM Best Params: {'kernel': 'rbf', 'gamma': 'scale', 'C': 10}
SVM Report:
              precision    recall  f1-score   support

         0.0       0.85      0.85      0.85       209
         1.0       0.87      0.87      0.87       241

    accuracy                           0.86       450
   macro avg       0.86      0.86      0.86       450
weighted avg       0.86      0.86      0.86       450

SVM Accuracy: 0.86
